# 04_119_LLM_호환확인

## EXAONE-4.0-1.2B + Flask API 호환성 확인

이 노트북은 프로젝트에 LLM 챗봇을 연결하기 전에 다음 항목을 순서대로 점검합니다.

1. Python / PyTorch / Transformers / Accelerate 버전
2. CPU / CUDA 사용 가능 여부
3. EXAONE 토크나이저 로드
4. EXAONE 모델 로드
5. 한국어 텍스트 생성
6. Flask `/api/dashboard-data` 연결
7. Flask `/api/realtime-beds` 연결
8. API 데이터를 EXAONE 컨텍스트로 전달하여 답변 생성

### 실행 환경
Jupyter에서 **EXAONE Chatbot** 커널을 선택한 뒤 실행하세요.

> Flask API 테스트를 할 때는 별도 CMD에서 프로젝트의 Flask 서버가 실행 중이어야 합니다.


## 1. 실행 환경 및 라이브러리 버전 확인

In [1]:
import sys
import torch
import transformers
import accelerate
import requests

print("Python       :", sys.version.split()[0])
print("PyTorch      :", torch.__version__)
print("Transformers :", transformers.__version__)
print("Accelerate   :", accelerate.__version__)
print("Requests     :", requests.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
print("실행 디바이스 :", "cuda" if torch.cuda.is_available() else "cpu")


C:\Users\Admin\anaconda3\envs\exaone_chatbot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python       : 3.10.22
PyTorch      : 2.14.1+cpu
Transformers : 5.18.0
Accelerate   : 1.15.0
Requests     : 2.34.2
CUDA 사용 가능: False
실행 디바이스 : cpu


## 2. EXAONE 토크나이저 호환 확인

In [2]:
from transformers import AutoTokenizer

MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"

print("토크나이저 로딩 중...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("토크나이저:", type(tokenizer).__name__)
print("EXAONE tokenizer OK")


토크나이저 로딩 중...
토크나이저: GPT2Tokenizer
EXAONE tokenizer OK


## 3. EXAONE 모델 호환 확인

CPU 환경의 메모리 사용량을 줄이기 위해 `torch_dtype="auto"`와 `low_cpu_mem_usage=True`를 사용합니다.


In [3]:
from transformers import AutoModelForCausalLM

print("EXAONE 모델 로딩 중...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    low_cpu_mem_usage=True,
)

model.eval()

print("EXAONE model OK")
print("dtype :", model.dtype)
print("device:", next(model.parameters()).device)


EXAONE 모델 로딩 중...


Loading weights: 100%|██████████| 332/332 [00:01<00:00, 288.28it/s]

EXAONE model OK
dtype : torch.bfloat16
device: cpu


## 4. 한국어 답변 생성 확인

`apply_chat_template()`에서 바로 텐서를 반환받지 않고, **문자열 프롬프트 생성 → 별도 토큰화** 순서로 처리합니다.


In [4]:
messages = [
    {
        "role": "user",
        "content": "서울시 응급의료 대시보드 챗봇 테스트입니다. 짧게 인사해주세요."
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt, return_tensors="pt")
inputs = {k: v.to(next(model.parameters()).device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
answer = tokenizer.decode(generated_ids, skip_special_tokens=True)

print("EXAONE 답변:")
print(answer.strip())


EXAONE 답변:
"안녕하세요! 서울시 응급의료 정보를 도와드릴게요 😊"


## 5. Flask API 기본 설정

현재 프로젝트 서버 기본 주소를 `http://127.0.0.1:5000`으로 설정합니다.


In [5]:
BASE_URL = "http://127.0.0.1:5000"

def api_get(path, params=None, timeout=10):
    url = BASE_URL.rstrip("/") + path
    response = requests.get(url, params=params, timeout=timeout)
    response.raise_for_status()
    return response.json()


## 6. 예측/분석 API 연결 확인

In [6]:
try:
    dashboard_data = api_get("/api/dashboard-data")
    print("상태: API 연결 성공")
    print("응답 타입:", type(dashboard_data).__name__)
    print("최상위 키:", list(dashboard_data.keys())[:20])
except Exception as e:
    print("API 연결 실패")
    print(type(e).__name__, ":", e)


상태: API 연결 성공
응답 타입: dict
최상위 키: ['meta', 'transport', 'population', 'relation', 'sample']


## 7. 실시간 병상 API 연결 확인

In [7]:
TEST_DISTRICT = "강남구"

try:
    bed_data = api_get(
        "/api/realtime-beds",
        params={"district": TEST_DISTRICT},
    )

    print("상태: API 연결 성공")
    print("자치구:", bed_data.get("district"))
    print("병원 수:", len(bed_data.get("hospitals", [])))
    print("기준시각:", bed_data.get("snapshot_updated_at"))

    for hospital in bed_data.get("hospitals", []):
        print(
            "-",
            hospital.get("Hospital_Name"),
            "| 가용병상:",
            hospital.get("Available_Beds"),
        )

except Exception as e:
    print("API 연결 실패")
    print(type(e).__name__, ":", e)


상태: API 연결 성공
자치구: 강남구
병원 수: 2
기준시각: 2026-10-02T14:43:14
- 연세대학교의과대학강남세브란스병원 | 가용병상: 7
- 삼성서울병원 | 가용병상: 0


## 8. API 데이터 기반 EXAONE 답변 테스트

이 단계에서는 모델의 자체 지식에 의존하지 않고, **Flask API에서 조회한 병상 데이터만 근거로 답변**하도록 시스템 프롬프트를 제공합니다.


In [8]:
def ask_exaone_with_context(question, context):
    context_text = json.dumps(context, ensure_ascii=False, indent=2)

    messages = [
        {
            "role": "system",
            "content": (
                "너는 서울시 응급 의료 및 이송 수요 예측 대시보드의 안내 챗봇이다.\n"
                "반드시 제공된 [API 조회 결과]만 근거로 답변해라.\n"
                "API에 없는 병원, 수치, 주소, 기능은 추측하거나 만들어내지 마라.\n"
                "Available_Beds가 null이면 '확인 불가'라고 설명해라.\n"
                "실시간 데이터라면 기준 시각을 함께 안내해라.\n"
                "짧고 명확한 한국어로 답변해라."
            ),
        },
        {
            "role": "user",
            "content": (
                f"사용자 질문:\n{question}\n\n"
                f"[API 조회 결과]\n{context_text}"
            ),
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt")
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()


In [9]:
import json

question = "현재 강남구 응급실 가용 병상 정보를 알려주세요."

try:
    context = api_get(
        "/api/realtime-beds",
        params={"district": "강남구"},
    )

    print("질문:", question)
    print("\nEXAONE 답변:")
    print(ask_exaone_with_context(question, context))

except Exception as e:
    print("테스트 실패")
    print(type(e).__name__, ":", e)


질문: 현재 강남구 응급실 가용 병상 정보를 알려주세요.

EXAONE 답변:
강남구 응급실 가용 병상 현황 (2026년 10월 2일 14:43시 기준):

1. 연세대학교의과대학강남세브란스병원: 7개 가용 병상
2. 삼성서울병원: 가용 병상 없음 (Overcrowded 상태)


## 9. 최종 호환성 체크

아래 셀은 현재 환경에서 핵심 구성요소가 정상인지 간단히 확인합니다.


In [10]:
checks = {
    "PyTorch import": bool(torch.__version__),
    "Transformers import": bool(transformers.__version__),
    "Accelerate import": bool(accelerate.__version__),
    "Requests import": bool(requests.__version__),
    "EXAONE tokenizer": tokenizer is not None,
    "EXAONE model": model is not None,
}

try:
    _ = api_get("/api/dashboard-data")
    checks["Dashboard API"] = True
except Exception:
    checks["Dashboard API"] = False

try:
    _ = api_get("/api/realtime-beds", {"district": "강남구"})
    checks["Realtime Beds API"] = True
except Exception:
    checks["Realtime Beds API"] = False

print("=== 호환성 확인 결과 ===")
for name, ok in checks.items():
    print(f"{'OK' if ok else 'FAIL':4} | {name}")

if all(checks.values()):
    print("\n모든 핵심 항목이 정상입니다.")
else:
    print("\nFAIL 항목을 확인해주세요.")


=== 호환성 확인 결과 ===
OK   | PyTorch import
OK   | Transformers import
OK   | Accelerate import
OK   | Requests import
OK   | EXAONE tokenizer
OK   | EXAONE model
OK   | Dashboard API
OK   | Realtime Beds API

모든 핵심 항목이 정상입니다.
